# Procurement Intelligence Assistant — MVP

RAG-based chatbot answering procurement, sourcing, and supply chain questions
from a curated set of YouTube video transcripts.

*Pipeline stages:* transcription (done) → chunking → embeddings → vector store → retrieval + LLM

## 1. Setup

In [1]:
# Core dependencies for chunking and token counting
import json
import tiktoken

# cl100k_base is the tokenizer used by GPT-3.5/4 family models —
# a reasonable proxy for token count even if using a different embedding model
enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text: str) ->int:
    return len(enc.encode(text))

## 2. Load Transcripts


#Load transcripts

Sourcedata: data/transcripts.json
Structure: {video_id: {title, segments: [{start, end, text}, ...]}}
#Two videos (uSjrTpJHn2g, U1E6rtnreoA) were pre-trimmed to remove webinar preamble/roll-call content before this file was saved.

In [4]:
with open("data/transcripts.json") as f:
    data = json.load(f)

print(f"Loaded {len(data)} videos")

Loaded 10 videos


## 3. Chunking

*Strategy:* merge consecutive Whisper segments into ~250-token chunks,
with ~40-token overlap so ideas that span a chunk boundary (e.g. a numbered
list) still appear complete in at least one chunk.

Each chunk keeps video_id, title, start, end as metadata —
needed later for citations and clickable timestamped YouTube links.

In [5]:
def count_tokens(text: str) -> int:
    """Return the token count of a string using the cl100k_base tokenizer."""
    return len(enc.encode(text))


def chunk_segments(segments, target_tokens=250, overlap_tokens=40):
    """
    Merge a list of {start, end, text} segments into token-bounded chunks.

    Args:
        segments: list of transcript segments for one video
        target_tokens: approx. token count to accumulate before closing a chunk
        overlap_tokens: approx. token count carried over into the next chunk

    Returns:
        list of {text, start, end, token_count} dicts
    """
    chunks = []
    current = []
    current_tokens = 0

    for seg in segments:
        current.append(seg)
        current_tokens += count_tokens(seg["text"])

        if current_tokens >= target_tokens:
            chunks.append(_build_chunk(current, current_tokens))
            current, current_tokens = _take_overlap(current, overlap_tokens)

    if current:
        chunks.append(_build_chunk(current, current_tokens))

    return chunks


def _build_chunk(seg_list, token_count):
    """Join a list of segments into a single chunk record."""
    return {
        "text": " ".join(s["text"].strip() for s in seg_list),
        "start": seg_list[0]["start"],
        "end": seg_list[-1]["end"],
        "token_count": token_count,
    }


def _take_overlap(seg_list, overlap_tokens):
    """Return the trailing segments (and their token count) to seed the next chunk."""
    overlap_seg_list = []
    overlap_count = 0
    for s in reversed(seg_list):
        overlap_count += count_tokens(s["text"])
        overlap_seg_list.insert(0, s)
        if overlap_count >= overlap_tokens:
            break
    return overlap_seg_list, overlap_count

## 4.Test chunk size in one video

In [6]:
test_video = "ZCIJQJRw6xw"

for target in [200, 250, 300]:
    test_chunks = chunk_segments(
        data[test_video]["segments"],
        target_tokens=target,
        overlap_tokens=int(target * 0.15)
    )
    print(f"\n=== target={target} tokens ({len(test_chunks)} chunks) ===")
    print(test_chunks[1]["text"])


=== target=200 tokens (13 chunks) ===
Whether it's software for a startup or machines for a factory, procurement decides who delivers what, when, and how much it costs. For example, think of procurement like planning a big wedding. You don't just go buy food and flowers. You plan your guest list, find reliable caterers, compare quotations, ensure everything arrives on time, track the budget, sign contracts. That's procurement on a business scale. Let's take a real-world example. Tata Motors, one of India's largest automobile manufacturers. To build each vehicle, they need tires from Bridgestone, steel from Tata Steel, electronics from Bosch, paint, seats, dashboards, software systems, etc. Tata Motors doesn't manufacture all of this in-house. Instead, they procure parts from a global network of suppliers who specialize in those products. If even one part, like a microchip, doesn't arrive on time, production halts, deadlines are missed, and millions are lost. That's how powerful procur

## 5. Lock in final size and run on all videos

In [8]:
FINAL_TARGET = 250   # <- update based on Step 4
FINAL_OVERLAP = 40    # <- update based on Step 4

all_chunks = []

for vid, content in data.items():
    video_chunks = chunk_segments(
        content["segments"],
        target_tokens=FINAL_TARGET,
        overlap_tokens=FINAL_OVERLAP
    )
    for c in video_chunks:
        c["video_id"] = vid
        c["title"] = content["title"]
        all_chunks.append(c)

print(f"Total chunks: {len(all_chunks)}")
print(f"Videos processed: {len(data)}")
print(f"Avg chunks per video: {len(all_chunks) / len(data):.1f}")

Total chunks: 160
Videos processed: 10
Avg chunks per video: 16.0


## 6. Sanity check the full set

In [9]:
import random

sample_chunks = random.sample(all_chunks, 5)

for c in sample_chunks:
    print(f"--- {c['title']} ({c['start']:.0f}s-{c['end']:.0f}s, {c['token_count']} tokens) ---")
    print(c["text"])
    print()

--- Contract Management in Procurement | Stages & Tools (129s-213s, 263 tokens) ---
Stage 4. Contract renewal or closure. Evaluate supplier performance and determine whether to extend or terminate the contract. Archive closed contracts and collect postmorm lessons. Example, a software company uses a 12-month SaaS subscription for a CRM tool. Near expiration, procurement evaluates usage, cost effectiveness, and vendor support to decide on renewal. Four, components of a procurement contract. Scope of work, SOW clear definition of the goods or services to be delivered. Pricing and payment terms include structure, fixed, milestone, hourly, due dates, and penalties. Service level agreements, SLAs, and KPIs, define expectations for delivery times, quality, and issue resolution. Termination clauses, outline under what circumstances the contract may be terminated early. Force majeure, provision to manage unforeseen disruptions, e.g. pandemics, natural disasters. Confidentiality and IP rights p

## 7.Save the chunks

In [10]:
with open("data/chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)

print("Saved", len(all_chunks), "chunks to data/chunks.json")

Saved 160 chunks to data/chunks.json
